# NOTEBOOK 02 – NGỌC LAN | CHƯƠNG 2: CƠ SỞ LÝ THUYẾT

**Phụ trách:** giữ nguyên 2.1–2.5, đồng thời thực hành huấn luyện **Random Forest** và đo **Precision, Recall, F1, AP/PR-AUC, FP, Top-K trên Validation**. Issue **#3**. Lý thuyết 12/11, RF bàn giao trước 20/11/2026.

**Mục đích:** tạo phần lý thuyết chuẩn, có thể giải thích khi bảo vệ; phụ trách mô hình Random Forest; **không dùng Test để chọn mô hình/ngưỡng**. Trúc Chi phụ trách Dummy Classifier và tích hợp mô hình; Tấn Lên huấn luyện MLP; Ngọc Lan huấn luyện Random Forest; Vĩnh Chi đánh giá độc lập cả ba mô hình trên Test.

[Tệp Markdown lý thuyết chi tiết](../docs/theory/02_ngoc_lan_ch2.md) · [Nguồn](../docs/REFERENCES.md).

## 2.1 Cơ sở về TMĐT và logistics
Mô tả vòng đời đơn đặt hàng → phê duyệt thanh toán → chuẩn bị hàng → giao cho vận chuyển → giao đến khách.

**Khái niệm:** `estimated delivery date` = ngày giao dự kiến; `actual delivery date` = ngày giao thực tế; `SLA` = thời hạn cam kết theo điều kiện dịch vụ.

**Tại sao dự báo sớm:** bộ phận vận hành cần ưu tiên theo dõi khi còn khả năng can thiệp. Việc phân biệt **dự báo trước** và **giải thích sau** là bắt buộc.

## 2.2 Khai thác dữ liệu và EDA
1. Kiểm tra schema, độ đầy đủ, khóa chính, khóa ngoại, quan hệ 1–n.
2. Chọn đơn vị phân tích **order_id** và tạo nhãn nhị phân bằng ngày lịch.
3. Thống kê đơn, đơn trễ, phân phối giá/phí vận chuyển, hành vi theo thời gian, bang và nhóm hàng.
4. Trực quan hóa có diễn giải: biểu đồ, bảng tỷ lệ, số mẫu của từng nhóm.
5. Phân biệt tương quan trong EDA và khả năng dự báo tại `order_approved_at`.

Cần liên kết hình và kết quả EDA do Lam Linh thực nghiệm, không tự đặt số liệu.

### 2.2.1. Thực hành T05 – Kiểm tra chất lượng dữ liệu thô

**Người thực hiện:** Ngọc Lan  
**Thành viên phối hợp:** Lam Linh  
**Nguồn dữ liệu:** Brazilian E-Commerce Public Dataset by Olist.

#### Mục tiêu

Kiểm tra chất lượng của 9 tệp CSV trước khi dữ liệu
được làm sạch, tích hợp và sử dụng để xây dựng mô hình
dự đoán nguy cơ giao hàng trễ.

Nội dung kiểm tra bao gồm:

1. Số dòng và số cột của từng bảng.
2. Số lượng và tỷ lệ giá trị thiếu (Missing Values).
3. Số lượng bản ghi trùng hoàn toàn (Duplicate Rows).
4. Kiểu dữ liệu của từng thuộc tính (Data Types).
5. Một số giá trị bất thường và tính hợp lệ của khóa.


#### Phương pháp

Sử dụng thư viện Pandas với các phương thức và thuộc tính:
`read_csv()`, `isna()`, `duplicated()`, `dtypes`,
`nunique()`, `describe()` và `value_counts()`.

Kết quả được thống kê theo từng bảng và từng cột,
làm cơ sở đánh giá chất lượng dữ liệu thô.

In [ ]:
# T05.1 - Thiết lập môi trường

from pathlib import Path
import pandas as pd
from IPython.display import display

# Tìm thư mục gốc repository
HERE = Path.cwd().resolve()

ROOT = next(
    (
        p for p in [HERE, *HERE.parents]
        if (p / "data" / "raw").is_dir()
    ),
    None
)

if ROOT is None:
    raise FileNotFoundError(
        "Không tìm thấy data/raw trong repository."
    )

RAW_DIR = ROOT / "data" / "raw"

FILES = {
    "orders": "olist_orders_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "translation": "product_category_name_translation.csv",
}

missing_files = [
    filename for filename in FILES.values()
    if not (RAW_DIR / filename).is_file()
]

if missing_files:
    raise FileNotFoundError(
        f"Thiếu dữ liệu: {missing_files}"
    )

print("Thư mục dự án:", ROOT)
print("Số file tìm thấy:", len(FILES))
print("Kết quả: Đã tìm thấy đủ 9 file CSV.")


In [ ]:
# T05.2 - Data Profiling toàn bộ 9 bảng

qa_summary = []
qa_columns = []

for table_name, filename in FILES.items():
    file_path = RAW_DIR / filename

    # Giữ các mã định danh và ZIP ở dạng string
    cols = pd.read_csv(file_path, nrows=0).columns

    dtype_map = {
        col: "string" for col in cols
        if (
            (col.endswith("_id")
             and col != "order_item_id")
            or "zip_code_prefix" in col
        )
    }

    data = pd.read_csv(
        file_path,
        dtype=dtype_map,
        low_memory=False
    )

    total_missing = int(data.isna().sum().sum())
    total_cells = data.size

    qa_summary.append({
        "table": table_name,
        "n_rows": len(data),
        "n_columns": len(data.columns),
        "missing_cells": total_missing,
        "missing_pct": round(
            100 * total_missing / total_cells, 3
        ),
        "duplicate_rows": int(
            data.duplicated().sum()
        ),
        "columns_with_missing": int(
            (data.isna().sum() > 0).sum()
        )
    })

    # Kiểm tra từng cột
    for col in data.columns:
        qa_columns.append({
            "table": table_name,
            "column": col,
            "dtype": str(data[col].dtype),
            "missing": int(data[col].isna().sum()),
            "missing_pct": round(
                100 * data[col].isna().mean(), 3
            ),
            "n_unique": int(
                data[col].nunique(dropna=True)
            )
        })

    print(f"Đã kiểm tra: {table_name}")

# Tổng hợp
qa_summary_df = pd.DataFrame(qa_summary)
qa_columns_df = pd.DataFrame(qa_columns)

assert len(qa_summary_df) == 9

print("\nBẢNG TỔNG HỢP QA - 9 CSV")
display(qa_summary_df)

print("\nTỔNG SỐ CỘT ĐÃ KIỂM TRA:",
      len(qa_columns_df))


In [ ]:
# T05.3 - Phân tích chi tiết theo từng cột

missing_detail = (
    qa_columns_df[
        qa_columns_df["missing"] > 0
    ]
    .sort_values(
        ["table", "missing"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

print("CÁC CỘT CÓ GIÁ TRỊ THIẾU")
display(missing_detail)

print("\nKIỂU DỮ LIỆU CỦA TẤT CẢ CÁC CỘT")
display(
    qa_columns_df[
        ["table", "column", "dtype", "n_unique"]
    ]
)

print("\nSỐ CỘT THEO KIỂU DỮ LIỆU")
display(
    qa_columns_df.groupby("dtype")
    .size()
    .reset_index(name="n_columns")
)


### 2.2.2. Nhận xét kết quả Data Profiling

Quá trình kiểm tra 9 bảng Olist ghi nhận:

- **Orders:** Có 4.908 ô thiếu, tập trung ở các trường
  thời gian phê duyệt, bàn giao vận chuyển và giao hàng.
  Các trường này cần được kiểm tra khi chọn mẫu và tạo nhãn.
- **Reviews:** Có 145.903 ô thiếu, chủ yếu ở tiêu đề
  và nội dung bình luận. Đây là các thông tin không bắt
  buộc, vì vậy không nên xóa toàn bộ bản ghi.
- **Products:** Có 2.448 ô thiếu liên quan đến danh mục,
  mô tả và thông số sản phẩm.
- **Geolocation:** Có 261.831 dòng trùng hoàn toàn.
  Cần xử lý phù hợp trước khi tổng hợp theo mã ZIP.

Các bảng còn lại không phát hiện giá trị thiếu hoặc
dòng trùng hoàn toàn trong bước profiling.

**Nhận xét:** Dữ liệu cần được xử lý theo ý nghĩa
nghiệp vụ của từng cột. Không xem mọi giá trị thiếu
hoặc khóa `order_id` lặp trong bảng chi tiết là lỗi.

**Đề xuất xử lý:**

- **Orders:** Kiểm tra thời gian thiếu trước khi chọn mẫu
  và xác định nhãn giao hàng trễ.
- **Reviews:** Không tự động loại bỏ bản ghi chỉ vì thiếu
  nội dung bình luận; không sử dụng đánh giá sau giao
  hàng làm đặc trưng dự báo.
- **Products:** Đánh giá giá trị thiếu theo từng thuộc tính
  trước khi quyết định điền thiếu hoặc loại bỏ.
- **Geolocation:** Kiểm tra trùng lặp và tổng hợp dữ liệu
  phù hợp theo ZIP prefix trước khi kết nối bảng.

Các đề xuất này được bàn giao cho Lam Linh để đối chiếu
với quy trình làm sạch trong Chương 3. Không thay đổi
dữ liệu gốc tại bước T05.

In [ ]:
# T05.4 - Kiểm tra khóa và giá trị bất thường

qa_checks = []

def add_check(table, check_name, affected, note):
    qa_checks.append({
        "table": table,
        "check": check_name,
        "affected_rows": int(affected),
        "note": note
    })

# 1. Kiểm tra order_id ở bảng orders
orders = pd.read_csv(
    RAW_DIR / FILES["orders"],
    dtype={"order_id": "string"}
)

add_check(
    "orders", "missing_order_id",
    orders["order_id"].isna().sum(),
    "Khóa đơn hàng không được thiếu"
)

add_check(
    "orders", "duplicate_order_id",
    orders["order_id"].duplicated().sum(),
    "order_id phải duy nhất trong bảng orders"
)

# 2. Kiểm tra bảng items
items = pd.read_csv(
    RAW_DIR / FILES["items"]
)

add_check(
    "items", "negative_price",
    (items["price"] < 0).sum(),
    "Giá sản phẩm âm là bất thường"
)

add_check(
    "items", "negative_freight",
    (items["freight_value"] < 0).sum(),
    "Phí vận chuyển âm là bất thường"
)

add_check(
    "items", "duplicate_item_key",
    items.duplicated(
        ["order_id", "order_item_id"]
    ).sum(),
    "Cặp order_id và order_item_id phải duy nhất"
)

# 3. Kiểm tra điểm đánh giá
reviews = pd.read_csv(
    RAW_DIR / FILES["reviews"]
)

invalid_review = (
    reviews["review_score"].notna()
    & ~reviews["review_score"].isin([1, 2, 3, 4, 5])
)

add_check(
    "reviews", "invalid_review_score",
    invalid_review.sum(),
    "Điểm đánh giá hợp lệ từ 1 đến 5"
)

# 4. Kiểm tra thanh toán
payments = pd.read_csv(
    RAW_DIR / FILES["payments"]
)

add_check(
    "payments", "negative_payment",
    (payments["payment_value"] < 0).sum(),
    "Giá trị thanh toán âm là bất thường"
)

add_check(
    "payments", "duplicate_payment_key",
    payments.duplicated(
        ["order_id", "payment_sequential"]
    ).sum(),
    "Khóa ghép thanh toán phải duy nhất"
)

# Tổng hợp
qa_checks_df = pd.DataFrame(qa_checks)

print("BẢNG KIỂM TRA GIÁ TRỊ BẤT THƯỜNG")
display(qa_checks_df)


In [ ]:
# T05.5 - Thống kê giá trị số và phân loại

print("THỐNG KÊ GIÁ SẢN PHẨM VÀ PHÍ VẬN CHUYỂN")
display(
    items[["price", "freight_value"]]
    .describe()
)

print("\nTHỐNG KÊ GIÁ TRỊ THANH TOÁN")
display(
    payments[["payment_value"]]
    .describe()
)

print("\nPHÂN PHỐI TRẠNG THÁI ĐƠN HÀNG")
display(
    orders["order_status"]
    .value_counts(dropna=False)
    .rename_axis("order_status")
    .reset_index(name="n_orders")
)

print("\nPHÂN PHỐI PHƯƠNG THỨC THANH TOÁN")
display(
    payments["payment_type"]
    .value_counts(dropna=False)
    .rename_axis("payment_type")
    .reset_index(name="n_records")
)


In [ ]:
# T05.6 - Xuất kết quả kiểm tra QA

OUTPUT_DIR = ROOT / "reports" / "tables"
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

qa_summary_df.to_csv(
    OUTPUT_DIR / "ngoc_lan_t05_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

qa_columns_df.to_csv(
    OUTPUT_DIR / "ngoc_lan_t05_columns.csv",
    index=False,
    encoding="utf-8-sig"
)

qa_checks_df.to_csv(
    OUTPUT_DIR / "ngoc_lan_t05_checks.csv",
    index=False,
    encoding="utf-8-sig"
)

print("ĐÃ XUẤT 3 FILE QA:")
print("1. ngoc_lan_t05_summary.csv")
print("2. ngoc_lan_t05_columns.csv")
print("3. ngoc_lan_t05_checks.csv")


### 2.2.3. Kết luận nhiệm vụ T05

Đã thực hiện Data Profiling trên đầy đủ 9 file CSV Olist,
kiểm tra 52 cột dữ liệu và tổng hợp kết quả về kích thước,
missing values, duplicate rows, kiểu dữ liệu và bất thường.

Kết quả cho thấy các vấn đề đáng chú ý tập trung ở
trường thời gian của orders, thông tin sản phẩm,
bình luận đánh giá và các dòng trùng của geolocation.

8 quy tắc kiểm tra khóa và giá trị bất thường
không phát hiện dòng vi phạm. Tuy nhiên, kết quả
này không đồng nghĩa dữ liệu đã hoàn toàn sạch.

Các kết quả QA được lưu thành bảng để đối chiếu với
Lam Linh. Việc làm sạch, tích hợp và tạo nhãn cuối
cùng thuộc quy trình xử lý dữ liệu của Chương 3.


## 2.3 Cơ sở thuật toán

**Dummy Classifier:** mốc tham chiếu để kiểm tra mô hình phức tạp có mang lại lợi ích hay không.

**Random Forest:** tập hợp nhiều cây quyết định bằng bootstrap và ngẫu nhiên hóa đặc trưng; phù hợp dữ liệu dạng bảng, có thể kiểm soát overfitting bằng min_samples_leaf/max_depth.

**MLP:** mạng nơ-ron nhiều lớp; học qua tối ưu hóa trọng số, cần mã hóa categorical và chuẩn hóa numeric, điều chỉnh architecture và regularization.

**Lưu ý học thuật:** không mặc định Deep Learning phải thắng Random Forest.

**Bổ sung thực hành 2.3:** Ngọc Lan trực tiếp `fit` Random Forest trên **Train chung** do giao thức Trúc Chi định nghĩa; xuất toàn bộ pipeline, xác suất RF trên Validation, cấu hình và metadata. Trúc Chi *không fit lại RF*; chỉ tải artifact và kiểm tra cùng `split_id`.


In [1]:
# ==================================================
# T15.1 - KIEM TRA DU LIEU DA SAN SANG CHO RANDOM FOREST
# Chi kiem tra - KHONG tao hoac thay doi split
# ==================================================

from pathlib import Path
import pandas as pd
import json
import hashlib
import sys

# Xac dinh thu muc goc cua repository
ROOT = next(
    (
        p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
        if (p / "docs" / "DATA_CONTRACT.md").exists()
    ),
    None
)

if ROOT is None:
    raise FileNotFoundError(
        "Khong tim thay thu muc goc repository."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.experiment import SAFE_FEATURES, PROTOCOL_VERSION

PROCESSED = ROOT / "data" / "processed"

required_files = [
    "orders_model_ready.csv",
    "feature_whitelist.json",
    "data_manifest.json"
]

print("THU MUC DU AN:", ROOT)
print("\n1. KIEM TRA FILE DA XU LY")

for filename in required_files:
    path = PROCESSED / filename
    print(
        f"{filename}:",
        "DA CO" if path.is_file() else "CHUA CO"
    )

assert all(
    (PROCESSED / filename).is_file()
    for filename in required_files
), "Thieu file dau vao cua Chuong 3."

# Doc bo du lieu model-ready
data_path = PROCESSED / "orders_model_ready.csv"
df = pd.read_csv(data_path)

print("\n2. THONG TIN DU LIEU")
print("So dong:", len(df))
print("So cot:", len(df.columns))

required_columns = [
    "order_id",
    "is_late",
    "order_approved_at",
    "order_delivered_customer_date"
] + SAFE_FEATURES

missing_columns = sorted(
    set(required_columns) - set(df.columns)
)

print("Cot bat buoc bi thieu:", missing_columns)

assert not missing_columns, "Du lieu thieu cot."
assert df["order_id"].notna().all(), "order_id bi thieu."
assert df["order_id"].is_unique, "order_id bi trung."
assert df["is_late"].notna().all(), "Nhan bi thieu."
assert df["is_late"].isin([0, 1]).all(), "Nhan khong hop le."

print("\n3. KIEM TRA FEATURE WHITELIST")

with open(
    PROCESSED / "feature_whitelist.json",
    encoding="utf-8"
) as f:
    whitelist = json.load(f)

print("Feature trong notebook:", SAFE_FEATURES)
print("Feature trong whitelist:", whitelist)

assert whitelist == SAFE_FEATURES, (
    "Danh sach feature khong trung voi giao thuc."
)

print("\n4. PHAN BO NHAN")
print(df["is_late"].value_counts().sort_index())

print("\n5. KIEM TRA MANIFEST")

with open(
    PROCESSED / "data_manifest.json",
    encoding="utf-8"
) as f:
    manifest = json.load(f)

actual_sha = hashlib.sha256(
    data_path.read_bytes()
).hexdigest()

print("SHA256 model-ready:", actual_sha)
print("SHA256 trong manifest:", manifest.get("model_sha256"))

assert actual_sha == manifest.get("model_sha256"), (
    "SHA256 khong khop: can doi chieu lai voi Lam Linh."
)

print("\n6. KIEM TRA SPLIT CHUNG")
print("Giao thuc:", PROTOCOL_VERSION)

for filename in ["split_manifest.csv", "split_metadata.json"]:
    print(
        filename,
        "DA CO" if (PROCESSED / filename).is_file()
        else "CHUA CO"
    )

print("\nPASS: Du lieu model-ready va whitelist hop le.")
print("Chua chia tap va chua huan luyen Random Forest.")


THU MUC DU AN: D:\UTH HK1 NAM 3\ĐO AN\Do_An_Khoa_hoc_du_lieu---GR02

1. KIEM TRA FILE DA XU LY
orders_model_ready.csv: DA CO
feature_whitelist.json: DA CO
data_manifest.json: DA CO

2. THONG TIN DU LIEU
So dong: 96456
So cot: 37
Cot bat buoc bi thieu: []

3. KIEM TRA FEATURE WHITELIST
Feature trong notebook: ['purchase_month_number', 'purchase_weekday', 'purchase_hour', 'approval_delay_hours', 'customer_state']
Feature trong whitelist: ['purchase_month_number', 'purchase_weekday', 'purchase_hour', 'approval_delay_hours', 'customer_state']

4. PHAN BO NHAN
is_late
0    89922
1     6534
Name: count, dtype: int64

5. KIEM TRA MANIFEST
SHA256 model-ready: c23d4c4d9fdcab86b73659a0273d676277a90a2dd077cbb079570d8b7673de7f
SHA256 trong manifest: c23d4c4d9fdcab86b73659a0273d676277a90a2dd077cbb079570d8b7673de7f

6. KIEM TRA SPLIT CHUNG
Giao thuc: gr02-temporal-v2
split_manifest.csv CHUA CO
split_metadata.json CHUA CO

PASS: Du lieu model-ready va whitelist hop le.
Chua chia tap va chua huan luye

In [3]:
# ==================================================
# T15.2 - KIEM TRA DU LIEU TRUOC KHI CHIA TAP
# Khong xoa dong, khong thay doi du lieu goc
# ==================================================

import pandas as pd
from IPython.display import display

check_df = df.copy()

# 1. Kiem tra khoa va nhan
assert check_df["order_id"].notna().all()
assert check_df["order_id"].is_unique
assert check_df["is_late"].isin([0, 1]).all()

# 2. Kiem tra cac moc thoi gian
approved = pd.to_datetime(
    check_df["order_approved_at"], errors="coerce"
)

delivered = pd.to_datetime(
    check_df["order_delivered_customer_date"],
    errors="coerce"
)

print("1. KIEM TRA THOI GIAN")
print("Thieu thoi diem duyet:", approved.isna().sum())
print("Thieu thoi diem giao:", delivered.isna().sum())

# Phat hien truong hop bat thuong
invalid_time = (
    approved.notna()
    & delivered.notna()
    & (delivered < approved)
)

n_invalid = int(invalid_time.sum())

print("So don giao truoc thoi diem duyet:", n_invalid)
print(
    "Ty le bat thuong:",
    round(n_invalid / len(check_df) * 100, 4),
    "%"
)

if n_invalid > 0:
    print("\nCAC DON CO MOC THOI GIAN BAT THUONG:")

    display(
        check_df.loc[
            invalid_time,
            [
                "order_id",
                "order_approved_at",
                "order_delivered_customer_date",
                "is_late"
            ]
        ].head(10)
    )

# 3. Kiem tra 5 dac trung
print("\n2. KIEM TRA FEATURE")

feature_report = pd.DataFrame({
    "feature": SAFE_FEATURES,
    "missing_count": [
        int(check_df[c].isna().sum())
        for c in SAFE_FEATURES
    ],
    "dtype": [
        str(check_df[c].dtype)
        for c in SAFE_FEATURES
    ]
})

display(feature_report)

# 4. Phan bo nhan
print("\n3. PHAN BO NHAN")

label_report = (
    check_df["is_late"]
    .value_counts()
    .sort_index()
    .rename_axis("is_late")
    .reset_index(name="so_don")
)

label_report["ty_le_%"] = (
    label_report["so_don"] / len(check_df) * 100
).round(2)

display(label_report)

# 5. Ket luan kiem tra
print("\n4. KET LUAN")

assert approved.notna().all(), "Thieu ngay duyet."
assert delivered.notna().all(), "Thieu ngay giao."

if n_invalid > 0:
    print(
        "WARNING: Phat hien moc thoi gian bat thuong."
    )
    print(
        "Can doi chieu voi Lam Linh, khong tu sua du lieu."
    )
else:
    print("PASS: Khong phat hien dao nguoc moc thoi gian.")

print("Da hoan thanh T15.2 - khong thay doi du lieu.")


1. KIEM TRA THOI GIAN
Thieu thoi diem duyet: 0
Thieu thoi diem giao: 0
So don giao truoc thoi diem duyet: 61
Ty le bat thuong: 0.0632 %

CAC DON CO MOC THOI GIAN BAT THUONG:


,order_id,order_approved_at,order_delivered_customer_date,is_late
193,58d4c4747ee059eeeb865b349b41f53a,2018-07-26 23:31:53,2018-07-25 23:58:19,0
466,4df92d82d79c3b52c7138679fa9b07fc,2018-07-29 23:30:52,2018-07-27 18:55:57,0
1932,6e57e23ecac1ae881286657694444267,2018-08-20 15:55:42,2018-08-17 16:45:45,0
3559,f222c56f035b47dfa1e069a88235d730,2018-02-04 23:31:47,2018-02-01 20:10:38,0
11406,cf72398d0690f841271b695bbfda82d2,2017-09-13 22:04:39,2017-09-11 14:15:02,0
13090,6df6c9c9af6ef75b4f06f8a7b9f47e9c,2018-07-28 23:31:06,2018-07-26 17:03:43,0
14146,1fab4ac9d85079b3da72a11475ae1685,2017-09-13 22:06:11,2017-09-08 20:13:03,0
15880,fa962e76e50f3469ae2abfa54e6d5be0,2018-07-27 23:30:58,2018-07-25 18:21:18,0
17580,0467205a89711e4ec8e70ef2277e3287,2018-07-05 16:40:30,2018-07-04 22:42:41,0
19958,ecd1661658d34312cf9ac2bf34ba3b6b,2018-07-05 16:25:45,2018-07-05 12:58:37,0



2. KIEM TRA FEATURE


,feature,missing_count,dtype
0,purchase_month_number,0,int64
1,purchase_weekday,0,int64
2,purchase_hour,0,int64
3,approval_delay_hours,0,float64
4,customer_state,0,object



3. PHAN BO NHAN


,is_late,so_don,ty_le_%
0,0,89922,93.23
1,1,6534,6.77



4. KET LUAN
Can doi chieu voi Lam Linh, khong tu sua du lieu.
Da hoan thanh T15.2 - khong thay doi du lieu.


In [4]:
# T15.2.1 - PHAN TICH 61 DON CO THOI GIAN BAT THUONG

abnormal_orders = check_df.loc[invalid_time].copy()

abnormal_orders["time_difference_hours"] = (
    (
        pd.to_datetime(
            abnormal_orders["order_delivered_customer_date"]
        )
        - pd.to_datetime(
            abnormal_orders["order_approved_at"]
        )
    ).dt.total_seconds() / 3600
)

print("Tong so don bat thuong:", len(abnormal_orders))

print("\n1. THONG KE CHENH LECH THOI GIAN")
display(
    abnormal_orders["time_difference_hours"].describe()
)

print("\n2. PHAN BO NHAN CUA 61 DON")
display(
    abnormal_orders["is_late"].value_counts()
)

print("\n3. CHI TIET 10 DON DAU")
display(
    abnormal_orders[
        [
            "order_id",
            "order_approved_at",
            "order_delivered_customer_date",
            "time_difference_hours",
            "is_late"
        ]
    ].head(10)
)

print("\nHoan thanh kiem tra. Khong sua du lieu.")


Tong so don bat thuong: 61

1. THONG KE CHENH LECH THOI GIAN


count     61.000000
mean     -39.054408
std       42.976017
min     -167.751667
25%      -55.826944
50%      -24.823333
75%       -3.452222
max       -0.088611
Name: time_difference_hours, dtype: float64


2. PHAN BO NHAN CUA 61 DON


is_late
0    60
1     1
Name: count, dtype: int64


3. CHI TIET 10 DON DAU


,order_id,order_approved_at,order_delivered_customer_date,time_difference_hours,is_late
193,58d4c4747ee059eeeb865b349b41f53a,2018-07-26 23:31:53,2018-07-25 23:58:19,-23.559444,0
466,4df92d82d79c3b52c7138679fa9b07fc,2018-07-29 23:30:52,2018-07-27 18:55:57,-52.581944,0
1932,6e57e23ecac1ae881286657694444267,2018-08-20 15:55:42,2018-08-17 16:45:45,-71.165833,0
3559,f222c56f035b47dfa1e069a88235d730,2018-02-04 23:31:47,2018-02-01 20:10:38,-75.352500,0
11406,cf72398d0690f841271b695bbfda82d2,2017-09-13 22:04:39,2017-09-11 14:15:02,-55.826944,0
13090,6df6c9c9af6ef75b4f06f8a7b9f47e9c,2018-07-28 23:31:06,2018-07-26 17:03:43,-54.456389,0
14146,1fab4ac9d85079b3da72a11475ae1685,2017-09-13 22:06:11,2017-09-08 20:13:03,-121.885556,0
15880,fa962e76e50f3469ae2abfa54e6d5be0,2018-07-27 23:30:58,2018-07-25 18:21:18,-53.161111,0
17580,0467205a89711e4ec8e70ef2277e3287,2018-07-05 16:40:30,2018-07-04 22:42:41,-17.963611,0
19958,ecd1661658d34312cf9ac2bf34ba3b6b,2018-07-05 16:25:45,2018-07-05 12:58:37,-3.452222,0



Hoan thanh kiem tra. Khong sua du lieu.


In [5]:
# ==================================================
# T15.3 - KIEM TRA SPLIT CHUNG CUA NHOM
# Khong tao moi hoac sua doi split
# ==================================================

import json
import pandas as pd
from IPython.display import display

split_manifest_path = PROCESSED / "split_manifest.csv"
split_metadata_path = PROCESSED / "split_metadata.json"

print("1. KIEM TRA FILE SPLIT")

manifest_exists = split_manifest_path.is_file()
metadata_exists = split_metadata_path.is_file()

print("split_manifest.csv:", manifest_exists)
print("split_metadata.json:", metadata_exists)

if manifest_exists and metadata_exists:

    split_manifest = pd.read_csv(split_manifest_path)

    with open(
        split_metadata_path,
        encoding="utf-8"
    ) as f:
        split_metadata = json.load(f)

    print("\n2. THONG TIN SPLIT")
    print("Split ID:", split_metadata.get("split_id"))
    print("Protocol:", split_metadata.get("protocol"))

    display(
        split_manifest["split"]
        .value_counts()
        .rename_axis("split")
        .reset_index(name="so_don")
    )

    print("\n3. KIEM TRA TRUNG LAP")

    assert split_manifest["order_id"].is_unique, (
        "Phat hien order_id trung trong manifest."
    )

    print("PASS: Manifest khong trung order_id.")

else:
    print("\nCHUA CO BO SPLIT CHUNG.")
    print(
        "Can nhan bo split da chot tu Truc Chi."
    )
    print(
        "Khong tu tao split truoc khi thong nhat."
    )


1. KIEM TRA FILE SPLIT
split_manifest.csv: False
split_metadata.json: False

CHUA CO BO SPLIT CHUNG.
Can nhan bo split da chot tu Truc Chi.
Khong tu tao split truoc khi thong nhat.


In [6]:
# T15.3.1 - KIEM TRA CAU HINH RANDOM FOREST
# Chua huan luyen, chua su dung Train/Test

import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

from src.experiment import (
    NUMERIC_FEATURES,
    CATEGORY_FEATURES,
    SAFE_FEATURES
)

def build_rf_pipeline(min_leaf=5):
    preprocess = ColumnTransformer([
        (
            "numeric",
            SimpleImputer(strategy="median"),
            NUMERIC_FEATURES
        ),
        (
            "categorical",
            Pipeline([
                ("impute", SimpleImputer(strategy="most_frequent")),
                ("ohe", OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=False
                ))
            ]),
            CATEGORY_FEATURES
        )
    ])

    model = RandomForestClassifier(
        n_estimators=200,
        min_samples_leaf=min_leaf,
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=42
    )

    return Pipeline([
        ("preprocess", preprocess),
        ("rf", model)
    ])

print("Scikit-learn:", sklearn.__version__)
print("Numeric features:", NUMERIC_FEATURES)
print("Categorical features:", CATEGORY_FEATURES)

for leaf in [2, 5, 10]:
    pipeline = build_rf_pipeline(min_leaf=leaf)
    print(
        f"RF min_samples_leaf={leaf}:",
        pipeline.get_params()["rf__n_estimators"],
        "trees"
    )

print("\nPASS: Da khoi tao 3 cau hinh RF.")
print("Chua huan luyen va chua danh gia.")


Scikit-learn: 1.9.1
Numeric features: ['purchase_month_number', 'purchase_weekday', 'purchase_hour', 'approval_delay_hours']
Categorical features: ['customer_state']
RF min_samples_leaf=2: 200 trees
RF min_samples_leaf=5: 200 trees
RF min_samples_leaf=10: 200 trees

PASS: Da khoi tao 3 cau hinh RF.
Chua huan luyen va chua danh gia.


## 2.4 Các độ đo đánh giá

Với lớp dương = đơn trễ:
- **Precision** = TP/(TP+FP): chất lượng cảnh báo.
- **Recall** = TP/(TP+FN): tỷ lệ đơn trễ được phát hiện.
- **F1** = 2PR/(P+R): điều hòa Precision và Recall.
- **Average Precision (AP)**: trung bình có trọng số của Precision theo mức tăng Recall trên đường Precision–Recall (`average_precision_score`).
- **False Positives**: số đơn báo trễ nhưng thực tế đúng hạn.
- **Precision@K**: tỷ lệ đơn thật sự trễ trong Top-K xác suất lớn nhất.
- **Recall@K**: tỷ lệ đơn trễ được phát hiện trong Top-K, so với tổng số đơn trễ.

Không đồng nhất AP với ROC-AUC. Có thể báo cả ROC-AUC phụ, nhưng AP được ưu tiên nếu lớp dương ít.
**Bài thực hành 2.4:** từ xác suất RF trên Validation tính lại TP, FP, FN, TN, Precision, Recall, F1, AP, Precision@K và Recall@K. **Chỉ minh họa thực nghiệm với Validation**; người làm Test độc lập là Vĩnh Chi.


In [ ]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score, confusion_matrix

# Ví dụ NHÂN TẠO để minh họa cách tính – KHÔNG phải kết quả Olist.
y_true = np.array([0, 1, 1, 0, 0, 1])
y_prob = np.array([0.10, 0.80, 0.60, 0.30, 0.70, 0.90])
threshold = 0.50
y_pred = (y_prob >= threshold).astype(int)
print("Precision minh họa:", round(precision_score(y_true, y_pred, zero_division=0), 3))
print("Recall minh họa:", round(recall_score(y_true, y_pred, zero_division=0), 3))
print("F1 minh họa:", round(f1_score(y_true, y_pred, zero_division=0), 3))
print("Average Precision minh họa:", round(average_precision_score(y_true, y_prob), 3))
print("Confusion matrix minh họa:\n", confusion_matrix(y_true, y_pred))

### 2.4.1 Gợi ý diễn giải hình minh họa
Giải thích mối đánh đổi khi thay đổi threshold: threshold thấp thường tăng Recall nhưng có thể tăng cảnh báo sai. **Kịch bản Top-K** thể hiện khả năng giám sát có giới hạn nhân lực. Ví dụ giả định ở trên chỉ để dạy thuật toán; bắt buộc xóa hoặc ghi rõ “minh họa” khi đưa vào slide báo cáo.
Phần ví dụ giả định trước đó chỉ để học công thức. Kết quả RF Validation có được từ các cell thực nghiệm tiếp theo, lưu riêng và có thể kiểm tra lại.


## THỰC HÀNH 2.3 – RANDOM FOREST (NGỌC LAN HUẤN LUYỆN)

**Đầu vào:** `data/processed/orders_model_ready.csv` từ Lam Linh; split Train/Validation/Test thống nhất và metadata từ `src/experiment.py`.

**Thời điểm dự báo:** `order_approved_at`. **Không dùng** `order_delivered_customer_date`, status cuối hoặc reviews làm feature.

**Trách nhiệm Ngọc Lan:** huấn luyện RF và chọn cấu hình bằng **AP Validation**, sau đó chọn threshold trên **Validation**, xuất `models/ngoc_lan_random_forest.joblib`, `rf_val_predictions.csv`, `rf_metadata.json`; bàn giao cho Trúc Chi. Không huấn luyện MLP/Dummy ở notebook này.

**Kiểm tra đầu tiên:** cần có `orders_model_ready.csv`; nếu Ch3 chưa bàn giao, dừng để yêu cầu Lam Linh hoàn thành. Nếu split đã được tạo, hệ thống kiểm tra SHA và `split_id` thay vì chia lại theo cách khác.

In [ ]:
from pathlib import Path
import sys, json, joblib
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
             if (p / "docs" / "DATA_CONTRACT.md").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Hãy mở notebook trong repo hoặc thư mục notebooks/")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
PROCESSED = ROOT / "data" / "processed"
from src.experiment import (NUMERIC_FEATURES, CATEGORY_FEATURES, SAFE_FEATURES,
                            get_shared_splits, choose_threshold_on_validation,
                            precision_recall_at_k)
splits = get_shared_splits(PROCESSED)
train, val = splits["train"], splits["val"]
split_id = splits["metadata"]["split_id"]
print("Split dùng chung:", split_id)
display(pd.DataFrame([{"split": k, "n": len(splits[k]),
                       "late_rate": splits[k]["is_late"].mean()}
                      for k in ("train", "val", "test")]))
print("Tuyệt đối không fit hoặc đánh giá RF trên Test trong notebook Ngọc Lan.")

### 2.3.1 Xây dựng pipeline Random Forest – Fit trên Train

Dùng cùng feature whitelist với Trúc Chi. Pipeline xử lý missing, one-hot categorical trước khi `RandomForestClassifier`.

**Tham số cần trình bày:** `n_estimators`, `min_samples_leaf`, `class_weight`, `random_state`; giải thích lý do và khả năng overfitting. Các cấu hình minh họa cần được chọn từ **Validation**.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score

X_train = train[SAFE_FEATURES]
y_train = train["is_late"].astype(int)
X_val = val[SAFE_FEATURES]
y_val = val["is_late"].astype(int)

def rf_pipeline(min_leaf=5):
    prep = ColumnTransformer([
        ("numeric", SimpleImputer(strategy="median"), NUMERIC_FEATURES),
        ("categorical", Pipeline([
            ("impute", SimpleImputer(strategy="most_frequent")),
            ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ]), CATEGORY_FEATURES)
    ])
    return Pipeline([
        ("preprocess", prep),
        ("rf", RandomForestClassifier(
            n_estimators=200, min_samples_leaf=min_leaf,
            class_weight="balanced_subsample", n_jobs=-1, random_state=42))
    ])

records, fitted = [], {}
for min_leaf in [2, 5, 10]:
    candidate = rf_pipeline(min_leaf)
    candidate.fit(X_train, y_train)  # TRAIN ONLY
    prob = candidate.predict_proba(X_val)[:, 1]  # VALIDATION ONLY
    score = float(average_precision_score(y_val, prob))
    records.append({"min_samples_leaf": min_leaf, "AP_validation": score})
    fitted[min_leaf] = candidate

cv_table = pd.DataFrame(records).sort_values(
    ["AP_validation", "min_samples_leaf"], ascending=[False, True])
display(cv_table)
best_leaf = int(cv_table.iloc[0]["min_samples_leaf"])
rf = fitted[best_leaf]
rf_val_prob = rf.predict_proba(X_val)[:, 1]
print("Đã chọn RF bằng AP trên Validation:", best_leaf, "split:", split_id)

### 2.4.1 Đánh giá Random Forest trên Validation

- Lớp dương: `is_late=1`.
- Chọn threshold bằng **F1 Validation**, nếu bằng nhau ưu tiên ít FP hơn.
- Báo **Precision, Recall, F1, AP, TP, FP, FN, TN** và so sánh Top-K.
- Kết quả ở đây chỉ là đánh giá Validation để phát triển phương pháp; **Vĩnh Chi** sẽ đánh giá chính thức trên frozen Test.

Chỉ đưa bảng giá trị thực sau khi chạy Olist. Không sao chép ví dụ nhân tạo thành kết quả nhóm.

In [ ]:
from src.experiment import classification_metrics
rf_val_metrics = choose_threshold_on_validation(y_val, rf_val_prob)
rf_threshold = float(rf_val_metrics["threshold"])
display(pd.DataFrame([{"model": "Random Forest", **rf_val_metrics,
                       "split_id": split_id, "dataset": "Validation"}]))
k_values = sorted(set(min(k, len(val)) for k in [50, 100, 200] if k > 0))
rf_topk_val = pd.DataFrame([
    precision_recall_at_k(y_val.to_numpy(), rf_val_prob, k)
    for k in k_values
])
display(rf_topk_val)
print("FP = số đơn cảnh báo sai trên Validation:",
      rf_val_metrics["FP"])

### 2.4.2 Bàn giao model RF cho Trúc Chi

**Bắt buộc:**
1. `models/ngoc_lan_random_forest.joblib`: pipeline RF đã fit bằng Train.
2. `data/processed/rf_val_predictions.csv`: `order_id`, `y_true`, `p_rf` của Validation.
3. `data/processed/rf_metadata.json`: fingerprint split, feature list, cấu hình, threshold và metrics trên Validation.
4. Có giải thích trong `reports/chapters/CH02_THEORY.md`; tách rõ thuật toán và kết quả thực nghiệm.
5. Link PR/commit và xác nhận Trúc Chi đã nhận. **Không đẩy CSV/model lớn lên GitHub** (bàn giao theo kênh nội bộ nhóm).

In [ ]:
MODEL_DIR = ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)
pred_val = pd.DataFrame({
    "order_id": val["order_id"].to_numpy(),
    "y_true": y_val.to_numpy(),
    "p_rf": rf_val_prob,
})
assert pred_val["order_id"].is_unique
pred_val.to_csv(PROCESSED / "rf_val_predictions.csv", index=False)
joblib.dump(rf, MODEL_DIR / "ngoc_lan_random_forest.joblib")

metadata = {
    "owner": "Ngọc Lan", "model": "rf", "split_id": split_id,
    "dataset_sha256": splits["metadata"]["csv_sha256"],
    "features": list(SAFE_FEATURES),
    "best_min_samples_leaf": best_leaf,
    "n_estimators": 200, "random_state": 42,
    "validation_metrics": rf_val_metrics,
    "validation_topk": rf_topk_val.to_dict(orient="records"),
    "validation_n": len(val),
    "threshold_origin": "Validation only"
}
(PROCESSED / "rf_metadata.json").write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("Đã lưu RF model, Validation predictions, metadata.")
print("Người nhận: Trúc Chi. Vĩnh Chi sẽ đánh giá Test độc lập.")

### Checklist bổ sung – Ngọc Lan
- [ ] RF chỉ fit trên Train chung; các bước encode/impute fit Train-only.
- [ ] Có bảng 3 cấu hình RF, chọn bằng AP Validation.
- [ ] Có Precision/Recall/F1/AP/FP/Top-K Validation.
- [ ] Đã bàn giao pipeline, `rf_val_predictions.csv`, `rf_metadata.json` với `split_id` và không dùng Test.
- [ ] Cập nhật GitHub Issue #3, GSheet mục 2.3–2.4 và T15.

## 2.5 Nghiên cứu liên quan
**Công việc:** lựa chọn **3–5 công trình** gần chủ đề giao hàng trễ, logistics/last-mile hoặc dự báo nhị phân trên dữ liệu bảng.

Bảng tổng quan bắt buộc có:
- Tác giả, năm và đường dẫn DOI/URL.
- Bộ dữ liệu và bối cảnh nghiên cứu.
- Mô hình, độ đo và đóng góp.
- Giới hạn và điểm khác so với đồ án của nhóm.

**Chưa có bài báo được nhóm xác minh** trong notebook này; không tự tạo tên tác giả hay kết quả từ nguồn chưa đọc.

In [ ]:
import pandas as pd
literature_template = pd.DataFrame(columns=[
    "tac_gia_nam", "tieu_de", "doi_url", "du_lieu", "mo_hinh", "do_do",
    "dong_gop", "han_che", "lien_he_de_tai"
])
display(literature_template)
print("Mẫu bảng, chưa điền tài liệu tham khảo chưa kiểm chứng.")

## 2.6 Hướng dẫn viết và nghiệm thu Chương 2
- [ ] Có định nghĩa/ký hiệu được giải thích bằng lời, không chép công thức thiếu nguồn.
- [ ] Thuật toán được mô tả **đúng với code cuối** của Trúc Chi.
- [ ] Các độ đo trùng định nghĩa và cách tính ở Chương 5 của Vĩnh Chi.
- [ ] Có 3–5 nguồn học thuật được tra cứu và xác minh.
- [ ] Mỗi mục liên kết cụ thể với đề tài Olist.
- [ ] Hoàn thành `reports/chapters/CH02_THEORY.md` và link Pull Request.

**Cấu trúc mỗi mục:** mở đầu – khái niệm – lý do sử dụng – ví dụ liên hệ Olist – hạn chế – tài liệu trích dẫn.
- [ ] Đã huấn luyện RF và có metrics Validation, artifact, metadata để Trúc Chi tích hợp.
- [ ] Không train RF trong notebook của Trúc Chi; Test được giữ kín cho Vĩnh Chi đánh giá cuối.
